# Bài 7 · Xử lý dữ liệu chuỗi

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Chuẩn hoá và lọc cột chuỗi, giữ rõ ý nghĩa của giá trị thiếu.
2. Giải thích một mẫu regex; phân biệt tìm, trích xuất và kiểm tra cả chuỗi.
3. Tách dữ liệu đa trị và tính tỷ lệ theo đúng đơn vị quan sát.
4. Kiểm chứng quy tắc bằng ví dụ khớp, không khớp và dữ liệu thật.

**Giáo trình:** [McKinney, §7.4 — String Manipulation](https://wesmckinney.com/book/data-cleaning#string-manipulation).
Các bảng nhỏ là **giả lập**. Dữ liệu thật dùng bản `data/listings.csv.gz`
của Santiago, snapshot **29/06/2026**; giá tính bằng **CLP/đêm**.
Code và output được kiểm bằng pandas **3.0.3**.

## Chuẩn bị

Ô cài đặt dành cho Colab. Sau đó chạy lần lượt các ô demo; các ô có nhãn
**Bài tập** để bạn tự viết lời giải.

In [ ]:
%pip -q install pandas==3.0.3 pyarrow==25.0.1

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

URL_FULL = ("https://data.insideairbnb.com/chile/rm/santiago/"
            "2026-06-29/data/listings.csv.gz")
tep = Path("data/listings_santiago_full_2026-06-29.csv.gz")
nguon = tep if tep.exists() else URL_FULL
df = pd.read_csv(nguon, usecols=[
    "id", "name", "price", "amenities", "neighbourhood_cleansed"
])
print("pandas:", pd.__version__)
print("Số chỗ ở:", len(df))

## 1. Chuẩn hoá và lọc chuỗi

Python xử lý từng chuỗi bằng các phương thức như `strip`, `lower`, `replace`, `split`.
pandas cung cấp `.str` để áp dụng các thao tác này cho một Series và giữ chỉ mục.
`.str` là cách viết thao tác trên cả cột; không có nghĩa mọi thao tác luôn nhanh hơn `map`.

### Xử lý chuỗi với .str

Với một chuỗi: `"  Ñuñoa ".strip().lower()`.

In [ ]:
s = pd.Series(["  Ñuñoa ", "ÑUÑOA", "ñuñoa", "Nunoa"])
chuan = s.str.strip().str.lower()
print(chuan.tolist())

`strip` bỏ khoảng trắng ở hai đầu; `lower` đổi sang chữ thường.

### Chuẩn hoá tên trước khi đếm

In [ ]:
print(chuan.value_counts())

`nunoa` vẫn khác `ñuñoa`. Sửa chính tả cần quy tắc riêng.

Giữ cột gốc để đối chiếu. Không mặc định xoá dấu, dấu câu hay khoảng trắng bên trong mọi chuỗi.

### Lọc tên có từ khoá

Lấy những tên có nhắc đến `metro`.

In [ ]:
ten = pd.Series(["Metro centro", "Casa con jardín", None])
mask = ten.str.contains(
    "metro", case=False, regex=False, na=False)
print(ten[mask].tolist())

`case=False`: bỏ qua hoa thường. `regex=False`: tìm nguyên chuỗi đã cho.

Regex sẽ được giải thích ở mục 2.

### Lọc chuỗi khi có giá trị thiếu

In [ ]:
s_thieu = pd.Series(["metro", None, "centro"], dtype="string")
kq = pd.DataFrame({
    "mac_dinh": s_thieu.str.contains("metro"),
    "na_False": s_thieu.str.contains("metro", na=False),
})
print(kq)

`na=False` loại tên thiếu khỏi kết quả lọc. Điều đó không chứng minh chỗ ở ấy không có metro.

### Đổi giá dạng chuỗi thành số

Trong nguồn này, `$` là ký hiệu giá bằng CLP; dấu phẩy phân cách hàng nghìn.

In [ ]:
gia_chu = (df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False))
df["price_num"] = pd.to_numeric(gia_chu, errors="coerce")
print(df[["price", "price_num"]].head(3))

`errors="coerce"` chuyển chuỗi không hợp lệ thành `NaN` (giá trị thiếu), thay vì báo lỗi.

Ví dụ: `"hỏi chủ"` → `NaN`.

### So sánh .str.replace với apply

Đo cùng việc bỏ `$` và dấu phẩy, giữ nguyên giá trị thiếu. Hai lần
`.str.replace(..., regex=False)` có thể viết thành một hàm rồi dùng `apply`.

**PyArrow** là thư viện Python của Apache Arrow: lưu dữ liệu theo cột và dùng
bộ xử lý viết bằng C++. pandas 3 ưu tiên dùng nó để lưu chuỗi khi đã cài;
`dtype="str"` không tự cho biết cột đang dùng PyArrow hay cách lưu Python.

Nguồn: [PyArrow](https://arrow.apache.org/docs/python/index.html),
[kiểu chuỗi từ pandas 3](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html).

In [ ]:
def bo_ky_hieu(x):
    if isinstance(x, str):
        return x.replace("$", "").replace(",", "")
    return x
gia_chu = df["price"].apply(bo_ky_hieu)

def thay_str(gia):
    return (gia.str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False))

def thay_apply(gia):
    return gia.apply(bo_ky_hieu)

pd.testing.assert_series_equal(
    thay_str(df["price"]), gia_chu, check_dtype=False)

Dùng cùng cột giá với hai cách lưu chuỗi. Chuyển kiểu **trước khi đo**,
để thời gian chuyển kiểu không lẫn vào thời gian thay chuỗi.

In [ ]:
import pyarrow as pa

kieu_arrow = pd.StringDtype("pyarrow", na_value=float("nan"))
cac_cot = {
    "PyArrow": df["price"].astype(kieu_arrow),
    "object": df["price"].astype(object),
}
print("pandas:", pd.__version__, "PyArrow:", pa.__version__)
print("Số giá:", len(df), "Thiếu:", df["price"].isna().sum())
print("Cách lưu cột price:", df["price"].dtype.storage)

Kiểm giá trị, chỉ mục và vị trí thiếu của hai kết quả trước khi đo.
Kiểu dữ liệu kết quả có thể khác, nên phép đối chiếu dưới đây bỏ qua dtype.
Thời gian chỉ gồm thay chuỗi và tạo Series kết quả; chưa gồm `to_numeric`.
Mỗi cách chạy lặp nhiều lần, lấy trung vị của chín lượt đo rồi tính ms/lần.
Số đo thay đổi theo máy và phiên bản thư viện; không đặt yêu cầu cách nào phải thắng.

In [ ]:
from statistics import median
from timeit import Timer

cac_cach = {".str.replace ×2": thay_str, "hàm + apply": thay_apply}
toc_do = []
for luu_chuoi, gia in cac_cot.items():
    kq_str, kq_apply = thay_str(gia), thay_apply(gia)
    pd.testing.assert_series_equal(kq_str, kq_apply, check_dtype=False)
    assert kq_str.isna().equals(gia.isna())
    for cach, fn in cac_cach.items():
        dong_ho = Timer(lambda fn=fn, gia=gia: fn(gia))
        so_lan, _ = dong_ho.autorange()
        mau_do = dong_ho.repeat(repeat=9, number=so_lan)
        toc_do.append({"luu_chuoi": luu_chuoi, "cach": cach,
                       "ms": median(mau_do) * 1000 / so_lan})

bang_toc_do = pd.DataFrame(toc_do).pivot(
    index="luu_chuoi", columns="cach", values="ms")
print(bang_toc_do.round(1))

Trong lần đo trên máy giảng viên (18.534 giá, pandas 3.0.3,
PyArrow 25.0.1, Python 3.12.3), kết quả là:

| Lưu chuỗi | Hai lần `.str.replace` | Hàm + `apply` |
|---|---:|---:|
| PyArrow | 2,7 ms | 13,0 ms |
| `object` | 13,1 ms | 11,1 ms |

Với phép thay literal này, `.str.replace` dùng bộ xử lý C++ của Arrow khi
cột lưu bằng PyArrow. `apply` gọi hàm Python cho từng giá trị. Với `object`,
`.str` vẫn xử lý các chuỗi Python; hai lần `.str.replace` tạo hai lượt xử lý
và kết quả trung gian, còn hàm gộp hai phép thay trong một lượt.

**Thao tác trên cả cột không bảo đảm luôn nhanh hơn. Kiểm kết quả rồi đo trên
đúng dữ liệu và cách lưu chuỗi đang dùng.** Không suy ra tốc độ của mọi hàm
`.str` hoặc mọi mẫu regex từ phép đo này.

### Phân biệt giá thiếu và lỗi chuyển đổi

In [ ]:
loi_doc = df["price"].notna() & df["price_num"].isna()
print(pd.Series({
    "thieu_tu_nguon": df["price"].isna().sum(),
    "khong_doc_duoc": loi_doc.sum(),
}))

Giữ giá gốc để phân biệt dữ liệu đã thiếu với lỗi do chuyển đổi.

### Bài tập 1: giá trung vị theo loại chỗ ở

Tính giá trung vị theo loại, gộp các nhãn chỉ khác hoa thường hoặc khoảng trắng.

Không điền giá thiếu bằng 0.

In [ ]:
bang = pd.DataFrame({
    "loai": [" Entire home ", "entire HOME", "PRIVATE ROOM",
             "Private room ", "private room"],
    "gia": [100, 300, 50, None, 150],
})
# TODO: tính giá trung vị theo loại đã chuẩn hoá.

## 2. Trích xuất bằng biểu thức chính quy

Cùng một thông tin có thể viết `1BR`, `3 bedrooms`, `2 br`.
Biểu thức chính quy (**regex**) mô tả mẫu: số → khoảng trắng nếu có → từ chỉ phòng ngủ.
Bài này xây quy tắc cho các cách viết trên; không xem nó là bộ đọc mọi mô tả phòng ngủ.

### Biểu thức chính quy (regex)

**Biểu thức chính quy** (*regular expression*, gọi tắt là **regex**) mô tả mẫu ký tự cần tìm trong chuỗi.

`.str.contains` kiểm tra trong mỗi chuỗi có đoạn khớp mẫu hay không. Ví dụ với `a.b` và `axb`:

In [ ]:
s_cham = pd.Series(["a.b", "axb"])
print(pd.DataFrame({
    "regex": s_cham.str.contains("."),
    "chuoi": s_cham.str.contains(".", regex=False),
}))

Trong regex, `.` khớp một ký tự bất kỳ, trừ xuống dòng theo mặc định.

### Các thành phần của mẫu regex

`r"(\d+)\s*(?:BR|bedrooms?)"`:

| Phần | Ý nghĩa |
|---|---|
| `\d+` | một hoặc nhiều chữ số |
| `\s*` | không có hoặc có nhiều ký tự trắng |
| <code>BR&#124;bedrooms?</code> | BR, bedroom hoặc bedrooms |
| `(\d+)` | `extract` trả về chữ số: `3 bedrooms` → `"3"`. |
| <code>(?:BR&#124;bedrooms?)</code> | Khớp từ chỉ phòng, nhưng `extract` không trả về phần chữ này. |

Với `3 bedrooms`, mẫu khớp cả cụm nhưng `extract` chỉ trả về `"3"`.
Từ `bedrooms` giúp nhận diện số phòng; nó không được đưa vào kết quả.

`r"..."` giữ dấu `\` cho regex.

### Lấy số phòng bằng extract

In [ ]:
ten_mau = pd.Series(["Modern 1BR Oasis", "Casa 3 bedrooms",
                    "12bravo centro", "Torre 2024"])
pat = r"(\d+)\s*(?:BR|bedrooms?)"
print(ten_mau.str.extract(pat, flags=re.I, expand=False))

`extract` lấy nhóm trong lần khớp đầu tiên. `12bravo` cũng khớp: mẫu còn quá rộng.

### Tránh khớp nhầm bằng ranh giới từ

In [ ]:
PAT_PHONG = r"\b(\d+)\s*(?:BR|bedrooms?)\b"
trich = ten_mau.str.extract(
    PAT_PHONG, flags=re.I, expand=False)
print(trich)

`\b` là ranh giới từ. Sau `BR` trong `bravo` vẫn là chữ, nên không khớp.

### Chuyển kết quả trích xuất sang số

In [ ]:
so_phong = pd.to_numeric(trich, errors="coerce")
print(so_phong.tolist())

Giá trị thiếu nghĩa là quy tắc này chưa lấy được số, không phải số phòng bằng 0.

### Kiểm tra mã bằng fullmatch

Đầu vào: `P12`, `P12x`, `mã P12`.

In [ ]:
ma = pd.Series(["P12", "P12x", "mã P12"])
print(pd.DataFrame({
    "co_doan_khop": ma.str.contains(r"P\d+", na=False),
    "khop_ca_chuoi": ma.str.fullmatch(r"P\d+", na=False),
}))

Tìm được một đoạn khớp không có nghĩa cả chuỗi đúng định dạng.

### Bài tập 2: trích diện tích

Viết mẫu trích diện tích để cho đúng kết quả trong bảng.

Chỉ xét số nguyên không dấu và đơn vị `m2` hoặc `m²` trong các cách viết trên.

In [ ]:
ten_dt = pd.Series(["Depto 45 m2", "Studio 30m²",
                    "80 M2 luxury", "Torre 2024", "30m2x"])
# TODO: trích diện tích và chuyển kết quả sang số.

### Trích số phòng từ dữ liệu Santiago

In [ ]:
trich_that = df["name"].str.extract(
    PAT_PHONG, flags=re.I, expand=False)
print(pd.Series({
    "so_ten": len(df),
    "khop_mau": trich_that.notna().sum(),
}))

- `flags=re.I`: bỏ qua hoa thường, khớp cả `BR`, `br` và `Br`. `re.I` là tên ngắn của `re.IGNORECASE`.
- `expand=False`: mẫu chỉ lấy nhóm `(\d+)` nên kết quả là **Series**, cùng chỉ mục với `df["name"]`. Mặc định `expand=True` trả về **DataFrame** một cột. Nếu mẫu có nhiều nhóm lấy kết quả, `expand=False` vẫn trả về DataFrame.

Mẫu chỉ khớp các cách viết `BR`, `bedroom(s)`. Cần xem cả tên khớp và tên không khớp.

Trước khi dùng cột mới, đọc nguyên văn một mẫu ở cả hai nhóm. Thêm từ khoá để tăng
số dòng khớp có thể làm tăng cả khớp nhầm; số dòng khớp không đo độ chính xác.

In [ ]:
for nhan, mask in [("Khớp", trich_that.notna()),
                  ("Không khớp", trich_that.isna())]:
    print(nhan)
    print(df.loc[mask, "name"].sample(n=5, random_state=7)
          .to_string(index=False))

`Apartment near metro` và `No metro nearby` đều chứa `metro`.
Đặt tên cột đúng ý nghĩa phép đo, như `nhac_metro`, thay vì `gan_metro`.
Việc giá khác nhau giữa hai nhóm tên không chứng minh ảnh hưởng của khoảng cách đến metro.

In [ ]:
df["nhac_metro"] = df["name"].str.contains(
    "metro", case=False, regex=False, na=False)
print(df.groupby("nhac_metro")["price_num"]
      .agg(["size", "count", "median"]))

## 3. Tách danh sách trong một ô

Với chuỗi có dấu phân cách đã biết, dùng `split`. Với một chuỗi JSON, dùng bộ đọc JSON.
Sau khi tách, mỗi tiện nghi vẫn phải gắn với mã chỗ ở.

### Tách chuỗi bằng dấu phân cách

In [ ]:
nhan = pd.Series(["Wifi;Kitchen", "Kitchen;Parking"],
                 index=["P1", "P2"])
print(nhan.str.split(";", regex=False))

Kết quả là một danh sách trong mỗi ô. Dấu `;` là quy ước của ví dụ này.

### Đọc danh sách bằng json.loads

In [ ]:
raw = pd.Series(['["Wifi", "Kitchen"]',
                 '["Kitchen", "Stove, gas"]', '[]'],
                index=["P1", "P2", "P3"])
ds = raw.map(json.loads)
print(ds)

`Stove, gas` là một giá trị. Cắt chuỗi JSON theo dấu phẩy sẽ làm sai danh sách.

### Tách tiện nghi thành từng hàng

`explode()` đưa mỗi phần tử trong danh sách thành một hàng, giữ chỉ mục của hàng gốc.


In [ ]:
tien_nghi = ds.explode()
print(tien_nghi)

`P1` và `P2` có hai tiện nghi nên mỗi mã xuất hiện hai lần. `P3` có danh sách rỗng
`[]`; sau `explode()` vẫn có một hàng `P3` với giá trị `NaN`.
`Stove, gas` vẫn là một tiện nghi.


### Bỏ hàng có giá trị thiếu

`dropna()` bỏ các hàng có giá trị thiếu trong Series và trả về Series mới.


In [ ]:
tien_nghi = tien_nghi.dropna()
print(tien_nghi)

Hàng `P3: NaN` bị bỏ khỏi `tien_nghi`. Các hàng còn lại vẫn giữ mã chỗ ở trong chỉ mục.
`dropna()` không loại chuỗi rỗng hay giá trị 0.

Có thể viết gọn hai bước thành `ds.explode().dropna()`. Dữ liệu gốc `ds` vẫn có
đủ ba chỗ ở, kể cả `P3`; cần dùng mẫu số này khi tính tỷ lệ chỗ ở có Wifi.


### Tính tỷ lệ chỗ ở có Wifi

Ví dụ có 3 chỗ ở và 4 mục tiện nghi.

In [ ]:
co_wifi = ds.map(lambda x: "Wifi" in x)
print(f"{co_wifi.sum()}/{len(ds)} chỗ ở có Wifi "
      f"({co_wifi.mean():.1%})")

Chia cho 4 sẽ tính tỷ lệ Wifi trong các mục tiện nghi được liệt kê.

### Bài tập 3: thống kê tiện nghi

Gộp nhãn chỉ khác hoa thường hoặc dấu cách ở hai đầu.

**Lập bảng số chỗ ở liệt kê từng tiện nghi.**


In [ ]:
ds_bt = pd.Series([
    ["Wifi", " wifi ", "Kitchen"],
    ["WIFI", "Kitchen"],
    ["Kitchen", "Parking", "Parking"],
    [],
], index=["P1", "P2", "P3", "P4"])
# TODO: lập bảng số chỗ ở liệt kê từng tiện nghi.


### Thống kê Wifi trong dữ liệu Santiago

In [ ]:
ds_that = df["amenities"].map(json.loads)
co_wifi_that = ds_that.map(lambda x: "Wifi" in x)
print(f"{co_wifi_that.sum():,}/{len(df):,} chỗ ở "
      f"({co_wifi_that.mean():.1%}) liệt kê Wifi")

Đếm các mục được liệt kê sau khi tách. Snapshot này không lặp cùng một tiện nghi trong
một danh sách, nên số lần xuất hiện cũng bằng số chỗ ở liệt kê tiện nghi đó.
Không mặc định điều này đúng với mọi nguồn dữ liệu.

In [ ]:
muc_that = ds_that.explode().dropna()
print(muc_that.value_counts().head(10))
print("Danh sách rỗng:", ds_that.map(len).eq(0).sum())

## Làm việc với AI thì sao?

AI có thể đề xuất mẫu và giải thích code. Cần nêu rõ định dạng, ví dụ phải khớp và
không được khớp. Tự kiểm các trường hợp ngoài ví dụ đã đưa vào prompt.
Với từ khoá, tên cột phải mô tả đúng điều quy tắc đo được.

## 📖 Tự học

### Bảng tra thao tác chuỗi

| Nhu cầu | Công cụ |
|---|---|
| Bỏ khoảng trắng ở hai đầu / đổi hoa thường | `strip`, `lower`, `upper`, `casefold` |
| Tìm chuỗi / tiền tố / hậu tố | `contains`, `startswith`, `endswith` |
| Thay / tách / nối | `replace`, `split`, `str.join` |
| Độ dài / lấy một đoạn | `str.len`, `str.slice` |
| Trích một / mọi lần khớp | `extract`, `extractall` |
| Kiểm từ đầu / cả chuỗi | `match`, `fullmatch` |

### Thêm ký hiệu regex

- `[0-9]`: một chữ số trong khoảng 0–9; `[abc]`: một ký tự thuộc tập.
- `\w`: ký tự từ (chữ, số hoặc `_`); `\s`: ký tự trắng.
- `+`, `*`, `?`, `{2,4}`: số lần lặp của phần ngay trước chúng.
- `^`, `$`: đầu và cuối chuỗi (cách xử lý xuống dòng phụ thuộc cờ).
- `\.`: dấu chấm thật; `.`: ký tự bất kỳ, trừ xuống dòng theo mặc định.

`re.search`, `re.findall`, `re.sub` dùng cùng kiểu mẫu trên một chuỗi Python,
không cần pandas. Dùng `fullmatch` khi cần kiểm toàn bộ định dạng.

In [ ]:
print(re.findall(r"\d+", "2 phòng, 3 giường"))
print(re.sub(r"\s+", " ", "Casa   centro\tmetro"))

### Giữ mọi lần khớp và số nguyên có giá trị thiếu

`extract` chỉ lấy lần khớp đầu tiên; `extractall` giữ mọi lần khớp.
`Int64` (chữ I viết hoa) là kiểu số nguyên cho phép thiếu.

In [ ]:
nhieu = pd.Series(["1BR main, 2BR annex", "Studio"])
print(nhieu.str.extractall(PAT_PHONG, flags=re.I))
print(so_phong.astype("Int64"))

### Kiểm cả định dạng giá

Quy tắc dưới chỉ nhận giá không âm bắt đầu bằng `$`, phần nguyên viết liền hoặc
có dấu phẩy sau mỗi ba chữ số, phần thập phân nếu có gồm đúng hai chữ số.
Chỉ đọc thành số sau khi kiểm định dạng; quy tắc này không xác minh đơn vị tiền.

In [ ]:
gia_test = pd.Series(["$1,200.00", "$850", "$1,20.00",
                      "giá $120.00", "$120.00abc", None])
PAT_GIA = r"\$(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d{2})?"
hop_le = gia_test.str.fullmatch(PAT_GIA, na=False)
print(pd.DataFrame({"gia": gia_test, "hop_le": hop_le}))

---

## Tóm tắt bài học

- Chuẩn hoá theo quy tắc; giữ cột gốc và ý nghĩa của giá trị thiếu.
- `contains` tìm, `extract` lấy nhóm, `fullmatch` kiểm toàn bộ chuỗi.
- Tách ô đa trị phải giữ liên hệ với đối tượng gốc và kiểm mẫu số.
- Quy tắc khớp chữ không tự xác minh ngữ nghĩa hay thực tế.

Chạy **Restart & Run all** sau khi hoàn thành các bài tập. Đối chiếu kết quả với
bảng nhỏ trước, rồi kiểm mẫu dữ liệu thật.

**Đọc thêm:** [McKinney §7.4](https://wesmckinney.com/book/data-cleaning#string-manipulation), [pandas: dữ liệu chuỗi](https://pandas.pydata.org/docs/user_guide/text.html),
[Python: regex](https://docs.python.org/3/howto/regex.html), [PyArrow](https://arrow.apache.org/docs/python/index.html), [pandas: kiểu chuỗi từ phiên bản 3](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html).

**Bài sau:** dữ liệu thời gian — đọc thời điểm, tính khoảng thời gian và tổng hợp theo kỳ.